In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from loaders._load_vn30_meta import _process_file, VN30, TARGETS
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_absolute_percentage_error, root_mean_squared_error

In [5]:
def preprocess(
    symbol: str,
    lag: int = 30,
    val: float = 0.0,
    verbose: bool = False,
):
    """
    Tiền xử lý dùng sai phân cho mô hình cây (không scale):
      - Feature X: các cột diff_lag_{1..lag} của 4 giá trị TARGETS trong {lag} ngày trước.
      - Target y: diff của ngày t+1 so với t (dịch -1).
      - Trả về kèm Y_base (giá trị gốc tại t) và Y_true (giá trị gốc tại t+1) để cộng ngược diff sau dự đoán.
      - Chia test theo đúng chiều dài df_test, đảm bảo số mẫu test == len(df_test).
    """
    # 1) Đọc & nối train/test, bỏ volume
    df_train, df_test = _process_file(symbol)
    df_train = df_train.drop(columns=['volume'])
    df_test  = df_test.drop(columns=['volume'])
    test_size = len(df_test)

    df_all = pd.concat([df_train, df_test], ignore_index=True)

    # 2) Sai phân cho từng cột mục tiêu
    for feat in TARGETS:
        df_all[f"{feat}_diff"] = df_all[feat].diff()

    # 3) Tạo X: lag trên *_diff
    diff_lag_cols = {}
    for feat in TARGETS:
        for i in range(1, lag + 1):
            diff_lag_cols[f"{feat}_diff_lag_{i}"] = df_all[f"{feat}_diff"].shift(i)
    X_block = pd.DataFrame(diff_lag_cols, index=df_all.index)

    # 4) Tạo y: diff của ngày t+1 (dịch -1)
    y_block = df_all[[f"{feat}_diff" for feat in TARGETS]].shift(-1)
    y_block.columns = [f"{feat}_y" for feat in TARGETS]  # open_y, high_y, low_y, close_y

    # 5) Tạo Y_base (giá trị gốc ở ngày t) và Y_true (giá trị gốc ở ngày t+1)
    Y_base_block = df_all[TARGETS].add_suffix('_base')          # ..._base = giá trị tại t
    Y_true_block = df_all[TARGETS].shift(-1).add_suffix('_true')# ..._true = giá trị tại t+1

    # 6) Ghép khung rồi dropna một lần
    frames = []
    if 'time' in df_all.columns:
        frames.append(df_all[['time']])  # chỉ để giữ index đồng bộ, sẽ drop sau
    frames.extend([X_block, y_block, Y_base_block, Y_true_block])
    work = pd.concat(frames, axis=1)

    if 'time' in work.columns:
        work = work.drop(columns=['time'])
    work = work.dropna()

    # 7) Tách X / y / Y_base / Y_true
    y_cols      = [f"{feat}_y" for feat in TARGETS]
    y_base_cols = [f"{feat}_base" for feat in TARGETS]
    y_true_cols = [f"{feat}_true" for feat in TARGETS]

    X = work.drop(columns=y_cols + y_base_cols + y_true_cols).values
    y = work[y_cols].values
    Y_base_all = work[y_base_cols].values   # (n_all, 4)
    Y_true_all = work[y_true_cols].values   # (n_all, 4)

    # 8) Chia train/test theo test_size (dòng cuối tương ứng các ngày test)
    X_train_full, X_test = X[:-test_size], X[-test_size:]
    y_train_full, y_test = y[:-test_size], y[-test_size:]
    Y_base_train_full, Y_base_test = Y_base_all[:-test_size], Y_base_all[-test_size:]
    Y_true_train_full, Y_true_test = Y_true_all[:-test_size], Y_true_all[-test_size:]

    # 9) Chia val từ phần train (nếu cần)
    n_samples = X_train_full.shape[0]
    valid_size = int(n_samples * val)
    train_size = n_samples - valid_size

    X_train, X_val   = X_train_full[:train_size], X_train_full[train_size:]
    y_train, y_val   = y_train_full[:train_size], y_train_full[train_size:]
    Y_base_train, Y_base_val = Y_base_train_full[:train_size], Y_base_train_full[train_size:]
    Y_true_train, Y_true_val = Y_true_train_full[:train_size], Y_true_train_full[train_size:]

    if verbose:
        print(f"=== Preprocessing (diff) {symbol} ===")
        print(f"Shapes X: train {X_train.shape}, val {X_val.shape}, test {X_test.shape}")
        print(f"Shapes y(diff): train {y_train.shape}, val {y_val.shape}, test {y_test.shape}")
        # Kiểm tra cột:
        assert y_train.shape[1] == len(TARGETS) == 4
        assert Y_true_test.shape[1] == len(TARGETS) == 4

    return {
        "train": (X_train, y_train),
        "val":   (X_val,   y_val),
        "test":  (X_test,  y_test),
        # Trả về luôn Y_base & Y_true (giá trị gốc) để cộng ngược diff sau khi dự đoán
        "Y_base": {"train": Y_base_train, "val": Y_base_val, "test": Y_base_test},
        "Y_true": {"train": Y_true_train, "val": Y_true_val, "test": Y_true_test},
    }

In [8]:
_ = preprocess('ACB', lag=30, verbose=True)

=== Preprocessing (diff) ACB ===
Shapes X: train (1213, 120), val (0, 120), test (328, 120)
Shapes y(diff): train (1213, 4), val (0, 4), test (328, 4)


In [17]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=7)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=DecisionTreeRegressor(),
        param_distributions={
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9357, MAPE: 0.8752
Symbol: BCM, R2: 0.9469, MAPE: 1.4203
Symbol: BID, R2: 0.8974, MAPE: 1.1047
Symbol: BVH, R2: 0.9735, MAPE: 1.1679
Symbol: CTG, R2: 0.9617, MAPE: 1.1833
Symbol: FPT, R2: 0.9884, MAPE: 1.2284
Symbol: GAS, R2: 0.9491, MAPE: 0.8060
Symbol: GVR, R2: 0.9607, MAPE: 1.8159
Symbol: HDB, R2: 0.9711, MAPE: 1.1603
Symbol: HPG, R2: 0.9033, MAPE: 1.0268
Symbol: LPB, R2: 0.9945, MAPE: 1.3751
Symbol: MBB, R2: 0.9500, MAPE: 1.1364
Symbol: MSN, R2: 0.9474, MAPE: 1.2169
Symbol: MWG, R2: 0.9796, MAPE: 1.3236
Symbol: PLX, R2: 0.9779, MAPE: 1.1769
Symbol: SAB, R2: 0.9126, MAPE: 0.9804
Symbol: SHB, R2: 0.9516, MAPE: 1.0041
Symbol: SSB, R2: 0.9538, MAPE: 0.9517
Symbol: SSI, R2: 0.9267, MAPE: 1.1705
Symbol: STB, R2: 0.9728, MAPE: 1.2187
Symbol: TCB, R2: 0.9760, MAPE: 1.1886
Symbol: TPB, R2: 0.9444, MAPE: 1.1756
Symbol: VCB, R2: 0.8502, MAPE: 0.7972
Symbol: VHM, R2: 0.9595, MAPE: 1.2464
Symbol: VIB, R2: 0.9256, MAPE: 0.9806
Symbol: VIC, R2: 0.9634, MAPE: 1.1704
Symbol: VJC,

# Random Forest

In [22]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=7)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=RandomForestRegressor(n_estimators=25, n_jobs=-1, random_state=42),
        param_distributions={
            "max_depth": [3, 5, 7, 9],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf": [1, 2, 4]
        }, 
        cv=tscv, 
        n_iter=10, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9357, MAPE: 0.8755
Symbol: BCM, R2: 0.9470, MAPE: 1.4150
Symbol: BID, R2: 0.9013, MAPE: 1.0957
Symbol: BVH, R2: 0.9734, MAPE: 1.1730
Symbol: CTG, R2: 0.9631, MAPE: 1.1562
Symbol: FPT, R2: 0.9887, MAPE: 1.2104
Symbol: GAS, R2: 0.9492, MAPE: 0.8160
Symbol: GVR, R2: 0.9641, MAPE: 1.7812
Symbol: HDB, R2: 0.9728, MAPE: 1.1386
Symbol: HPG, R2: 0.9041, MAPE: 1.0233
Symbol: LPB, R2: 0.9944, MAPE: 1.3709
Symbol: MBB, R2: 0.9477, MAPE: 1.1301
Symbol: MSN, R2: 0.9473, MAPE: 1.2181
Symbol: MWG, R2: 0.9798, MAPE: 1.3114
Symbol: PLX, R2: 0.9778, MAPE: 1.1788
Symbol: SAB, R2: 0.9213, MAPE: 0.9481
Symbol: SHB, R2: 0.9555, MAPE: 0.9879
Symbol: SSB, R2: 0.9525, MAPE: 0.9568
Symbol: SSI, R2: 0.9297, MAPE: 1.1503
Symbol: STB, R2: 0.9747, MAPE: 1.1837
Symbol: TCB, R2: 0.9769, MAPE: 1.1656
Symbol: TPB, R2: 0.9449, MAPE: 1.1649
Symbol: VCB, R2: 0.8599, MAPE: 0.7815
Symbol: VHM, R2: 0.9576, MAPE: 1.2361
Symbol: VIB, R2: 0.9233, MAPE: 0.9706
Symbol: VIC, R2: 0.9624, MAPE: 1.1757
Symbol: VJC,

# Diversity-Driven Forest

In [19]:
# ===== 1) Huấn luyện 1 cây với subspace & random search =====
def fit_one_tree(X_tr, y_tr, feature_idx, n_splits=3, random_state=42):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    est = DecisionTreeRegressor(random_state=random_state)
    search = RandomizedSearchCV(
        estimator=est,
        param_distributions={
            "max_depth": [3,5,7,9],
            "min_samples_split": [2,5,10],
            "min_samples_leaf": [1,2,4],
        },
        cv=tscv, n_iter=10, random_state=random_state
    )
    search.fit(X_tr[:, feature_idx], y_tr)
    return search.best_estimator_, feature_idx

# ===== 2) Xây pool cây =====
def build_pool(X_tr, y_tr, X_val, M=64, mtry=None, seed=42):
    rng = np.random.default_rng(seed)
    d = X_tr.shape[1]
    mtry = mtry or max(1, int(np.sqrt(d)))
    pool = []
    for m in range(M):
        feat_idx = rng.choice(d, size=mtry, replace=False)
        tree, feat_idx = fit_one_tree(X_tr, y_tr, feat_idx, random_state=seed+m)
        yhat_val = tree.predict(X_val[:, feat_idx])                 # (n_val, 4)
        pool.append({"tree": tree, "feat": feat_idx, "yhat_val": yhat_val})
    return pool

# ===== 3) Greedy chọn tập con đa dạng (unweighted) =====
def _vec_center(Y):
    v = Y.reshape(-1)
    return v - v.mean()

def _safe_corr(a, b, eps=1e-12):
    sa, sb = a.std(), b.std()
    if sa < eps or sb < eps: return 0.0
    return float(np.corrcoef(a, b)[0,1])

def select_diverse_unweighted(pool, Y_val, k=16, beta=0.5):
    n = len(pool)
    H = [_vec_center(p["yhat_val"]) for p in pool]
    y = _vec_center(Y_val)

    # start: best RMSE single
    rmses = [root_mean_squared_error(Y_val, pool[i]["yhat_val"]) for i in range(n) ]
    S = [int(np.argmin(rmses))]

    while len(S) < min(k, n):
        best_c, best_obj = None, None
        # current average
        avg = sum(pool[j]["yhat_val"] for j in S) / len(S)
        for c in range(n):
            if c in S: continue
            avg_c = (avg*len(S) + pool[c]["yhat_val"]) / (len(S)+1)
            rmse_c = root_mean_squared_error(Y_val, avg_c)
            # mean |corr| with selected
            corr_c = np.mean([abs(_safe_corr(H[c], H[j])) for j in S]) if S else 0.0
            obj = rmse_c + beta * corr_c
            if (best_obj is None) or (obj < best_obj):
                best_obj, best_c = obj, c
        S.append(best_c)

    return S  # indices in pool

# ===== 4) Tính trọng số (tùy chọn) =====
def stack_weights_with_diversity(pool, S, Y_val, lam=0.1, nonneg=True, sum_to_one=True):
    # Build H_S (centered) & y (centered)
    H_cols = [ _vec_center(pool[i]["yhat_val"]) for i in S ]
    H = np.column_stack(H_cols)                 # (n_val*d, k)
    y = _vec_center(Y_val)                      # (n_val*d,)
    # correlation matrix C_S
    k = len(S)
    C = np.eye(k)
    for i in range(k):
        for j in range(i+1, k):
            C[i,j] = C[j,i] = _safe_corr(H[:,i], H[:,j])

    # ridge-like closed form, then project
    A = H.T @ H + lam * C
    b = H.T @ y
    w = np.linalg.solve(A + 1e-8*np.eye(k), b)

    # optional constraints
    if nonneg:
        w = np.maximum(w, 0.0)
    if sum_to_one:
        s = w.sum()
        w = w / s if s > 1e-12 else np.ones_like(w)/k
    return w  # shape (k,)

# ===== 5) Suy luận trên test =====
def predict_ensemble(pool, S, X_test, Y_base_test, weights=None):
    yhats = []
    for i in S:
        tree, feat = pool[i]["tree"], pool[i]["feat"]
        yhats.append(tree.predict(X_test[:, feat]))           # (n_test, 4)
    yhats = np.stack(yhats, axis=0)                           # (k, n_test, 4)
    if weights is None:
        y_pred_diff = yhats.mean(axis=0)
    else:
        w = np.asarray(weights).reshape(-1,1,1)               # (k,1,1)
        y_pred_diff = (w * yhats).sum(axis=0)
    return Y_base_test + y_pred_diff

## Không học trọng số

In [25]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    # 0) Lấy dữ liệu
    data = preprocess(symbol, lag=7, val=0.1)
    X_train, y_train = data["train"]
    X_val,   y_val   = data["val"]
    X_test,  y_test  = data["test"]  # (diff) — chỉ dùng để tham khảo
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    # 1) Xây pool (M cây)
    pool = build_pool(X_train, y_train, X_val, M=50, mtry=None, seed=42)

    # 2) Chọn tập con đa dạng (k cây) – unweighted
    S = select_diverse_unweighted(pool, y_val, k=25, beta=1)

    # # 3a) Không trọng số
    Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=None)

    # 3b) (tuỳ chọn) Có trọng số với phạt đa dạng
    # w = stack_weights_with_diversity(pool, S, y_val, lam=0.1, nonneg=True, sum_to_one=True)
    # Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=w)

    # 4) Đánh giá toàn cục (multioutput)
    r2   = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100
    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9360, MAPE: 0.8681
Symbol: BCM, R2: 0.9477, MAPE: 1.4121
Symbol: BID, R2: 0.8992, MAPE: 1.1006
Symbol: BVH, R2: 0.9736, MAPE: 1.1698
Symbol: CTG, R2: 0.9633, MAPE: 1.1483
Symbol: FPT, R2: 0.9889, MAPE: 1.1981
Symbol: GAS, R2: 0.9501, MAPE: 0.8094
Symbol: GVR, R2: 0.9639, MAPE: 1.7809
Symbol: HDB, R2: 0.9727, MAPE: 1.1357
Symbol: HPG, R2: 0.9040, MAPE: 1.0269
Symbol: LPB, R2: 0.9944, MAPE: 1.3689
Symbol: MBB, R2: 0.9500, MAPE: 1.1202
Symbol: MSN, R2: 0.9469, MAPE: 1.2218
Symbol: MWG, R2: 0.9803, MAPE: 1.3019
Symbol: PLX, R2: 0.9777, MAPE: 1.1805
Symbol: SAB, R2: 0.9220, MAPE: 0.9451
Symbol: SHB, R2: 0.9556, MAPE: 0.9909
Symbol: SSB, R2: 0.9541, MAPE: 0.9324
Symbol: SSI, R2: 0.9290, MAPE: 1.1529
Symbol: STB, R2: 0.9746, MAPE: 1.1775
Symbol: TCB, R2: 0.9768, MAPE: 1.1595
Symbol: TPB, R2: 0.9457, MAPE: 1.1599
Symbol: VCB, R2: 0.8582, MAPE: 0.7796
Symbol: VHM, R2: 0.9579, MAPE: 1.2330
Symbol: VIB, R2: 0.9265, MAPE: 0.9586
Symbol: VIC, R2: 0.9630, MAPE: 1.1707
Symbol: VJC,

## Học trọng số

In [33]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    # 0) Lấy dữ liệu
    data = preprocess(symbol, lag=7, val=0.1)
    X_train, y_train = data["train"]
    X_val,   y_val   = data["val"]
    X_test,  y_test  = data["test"]  # (diff) — chỉ dùng để tham khảo
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    # 1) Xây pool (M cây)
    pool = build_pool(X_train, y_train, X_val, M=50, mtry=None, seed=42)

    # 2) Chọn tập con đa dạng (k cây) – unweighted
    S = select_diverse_unweighted(pool, y_val, k=25, beta=1)

    # # 3a) Không trọng số
    # Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=None)

    # 3b) (tuỳ chọn) Có trọng số với phạt đa dạng
    w = stack_weights_with_diversity(pool, S, y_val, lam=0.1, nonneg=True, sum_to_one=True)
    Y_pred = predict_ensemble(pool, S, X_test, Y_base_test, weights=w)

    # 4) Đánh giá toàn cục (multioutput)
    r2   = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100
    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")
    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9347, MAPE: 0.8817
Symbol: BCM, R2: 0.9482, MAPE: 1.4122
Symbol: BID, R2: 0.8909, MAPE: 1.1338
Symbol: BVH, R2: 0.9734, MAPE: 1.1726
Symbol: CTG, R2: 0.9619, MAPE: 1.1600
Symbol: FPT, R2: 0.9888, MAPE: 1.2050
Symbol: GAS, R2: 0.9493, MAPE: 0.8131
Symbol: GVR, R2: 0.9634, MAPE: 1.7934
Symbol: HDB, R2: 0.9725, MAPE: 1.1435
Symbol: HPG, R2: 0.9039, MAPE: 1.0202
Symbol: LPB, R2: 0.9942, MAPE: 1.3944
Symbol: MBB, R2: 0.9495, MAPE: 1.1260
Symbol: MSN, R2: 0.9470, MAPE: 1.2198
Symbol: MWG, R2: 0.9798, MAPE: 1.3145
Symbol: PLX, R2: 0.9773, MAPE: 1.1887
Symbol: SAB, R2: 0.9221, MAPE: 0.9442
Symbol: SHB, R2: 0.9560, MAPE: 0.9857
Symbol: SSB, R2: 0.9542, MAPE: 0.9365
Symbol: SSI, R2: 0.9287, MAPE: 1.1580
Symbol: STB, R2: 0.9747, MAPE: 1.1836
Symbol: TCB, R2: 0.9765, MAPE: 1.1607
Symbol: TPB, R2: 0.9459, MAPE: 1.1587
Symbol: VCB, R2: 0.8548, MAPE: 0.7825
Symbol: VHM, R2: 0.9581, MAPE: 1.2319
Symbol: VIB, R2: 0.9256, MAPE: 0.9729
Symbol: VIC, R2: 0.9631, MAPE: 1.1671
Symbol: VJC,

# XGBoost

In [27]:
from xgboost import XGBRegressor

tracks = {"r2": [], "mape": []}
for symbol in VN30:
    data = preprocess(symbol, lag=7)
    X_train, Y_train = data["train"]
    X_test, Y_test = data["test"]
    Y_base_test = data["Y_base"]["test"]
    Y_true_test = data["Y_true"]["test"]

    tscv = TimeSeriesSplit(n_splits=3)
    model = RandomizedSearchCV(
        estimator=XGBRegressor(n_estimators=25, n_jobs=-1, random_state=42),
        param_distributions={
            "max_depth": [3, 5, 7, 9]
        }, 
        cv=tscv, 
        n_iter=4, 
        random_state=42
    )

    model.fit(X_train, Y_train)
    Y_pred_diff = model.predict(X_test)
    Y_pred = Y_base_test + Y_pred_diff

    r2 = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9260, MAPE: 0.9389
Symbol: BCM, R2: 0.9432, MAPE: 1.5106
Symbol: BID, R2: 0.8943, MAPE: 1.1467
Symbol: BVH, R2: 0.9708, MAPE: 1.2318
Symbol: CTG, R2: 0.9600, MAPE: 1.2269
Symbol: FPT, R2: 0.9879, MAPE: 1.2742
Symbol: GAS, R2: 0.9469, MAPE: 0.8338
Symbol: GVR, R2: 0.9598, MAPE: 1.8891
Symbol: HDB, R2: 0.9708, MAPE: 1.2019
Symbol: HPG, R2: 0.8959, MAPE: 1.0805
Symbol: LPB, R2: 0.9939, MAPE: 1.4593
Symbol: MBB, R2: 0.9448, MAPE: 1.1686
Symbol: MSN, R2: 0.9417, MAPE: 1.2855
Symbol: MWG, R2: 0.9781, MAPE: 1.3858
Symbol: PLX, R2: 0.9769, MAPE: 1.2042
Symbol: SAB, R2: 0.9147, MAPE: 1.0157
Symbol: SHB, R2: 0.9535, MAPE: 1.0355
Symbol: SSB, R2: 0.9448, MAPE: 1.0675
Symbol: SSI, R2: 0.9246, MAPE: 1.2148
Symbol: STB, R2: 0.9722, MAPE: 1.2491
Symbol: TCB, R2: 0.9756, MAPE: 1.2031
Symbol: TPB, R2: 0.9380, MAPE: 1.2593
Symbol: VCB, R2: 0.8472, MAPE: 0.8325
Symbol: VHM, R2: 0.9540, MAPE: 1.2903
Symbol: VIB, R2: 0.9163, MAPE: 1.0302
Symbol: VIC, R2: 0.9593, MAPE: 1.2298
Symbol: VJC,

# Posterior-Sampling Forest

In [2]:
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_percentage_error

# ---------- 1) Huấn luyện 1 cây với subspace + random search ----------
def fit_one_tree(X_tr, y_tr, feat_idx, n_splits=3, seed=42):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    est = DecisionTreeRegressor(random_state=seed)
    search = RandomizedSearchCV(
        estimator=est,
        param_distributions={
            "max_depth": [3,5,7,9],
            "min_samples_split": [2,5,10],
            "min_samples_leaf": [1,2,4],
        },
        cv=tscv, n_iter=10, random_state=seed
    )
    search.fit(X_tr[:, feat_idx], y_tr)  # multi-output OK
    return search.best_estimator_, feat_idx

# ---------- 2) Xây pool ----------
def build_pool(X_tr, y_tr, X_val, M=64, mtry=None, seed=42):
    rng = np.random.default_rng(seed)
    d = X_tr.shape[1]
    mtry = mtry or max(1, int(np.sqrt(d)))
    pool = []
    for m in range(M):
        feat_idx = rng.choice(d, size=mtry, replace=False)
        tree, feat_idx = fit_one_tree(X_tr, y_tr, feat_idx, seed=seed+m)
        yhat_val = tree.predict(X_val[:, feat_idx])  # (n_val, n_targets)
        pool.append({"tree": tree, "feat": feat_idx, "yhat_val": yhat_val})
    return pool

# ---------- 3) Prior P (uniform / đa dạng / phức tạp) ----------
def _vec_center(Y): v = Y.reshape(-1); return v - v.mean()

def _safe_corr(a, b, eps=1e-12):
    sa, sb = a.std(), b.std()
    if sa < eps or sb < eps: return 0.0
    return float(np.corrcoef(a, b)[0,1])

def prior_uniform(M): 
    P = np.full(M, 1.0/M, dtype=float); return P

def prior_diversity(pool, alpha=0.5):
    M = len(pool)
    H = [ _vec_center(p["yhat_val"]) for p in pool ]
    redund = np.zeros(M, dtype=float)
    for i in range(M):
        ci = 0.0
        for j in range(M):
            if i==j: continue
            ci += abs(_safe_corr(H[i], H[j]))
        redund[i] = ci / max(1, M-1)
    logp = -alpha * redund
    logp -= logp.max()
    p = np.exp(logp); p /= p.sum()
    return p

def prior_complexity(pool, gamma=1.0):
    # penalize #leaves (or depth), keep closed-form
    leaves = np.array([p["tree"].get_n_leaves() for p in pool], dtype=float)
    logp = -gamma * (leaves - leaves.min()) / max(1.0, (leaves.max()-leaves.min()))
    logp -= logp.max()
    p = np.exp(logp); p /= p.sum()
    return p

def prior_combo(pool, alpha=0.5, gamma=0.5):
    p_div = prior_diversity(pool, alpha=alpha)
    p_cpx = prior_complexity(pool, gamma=gamma)
    p = p_div * p_cpx
    p /= p.sum()
    return p

# ---------- 4) Gibbs posterior Q* (nghiệm đóng) ----------
def pac_bayes_weights(losses, prior, lam=10.0):
    """
    losses: shape (M,), e.g., RMSE on val per tree
    prior:  shape (M,), sum(prior)=1
    lam:    temperature (lambda > 0)
    """
    # Stabilize via log-sum-exp
    logw = np.log(prior + 1e-16) - lam * (losses - losses.min())
    logw -= logw.max()
    w = np.exp(logw)
    w /= w.sum()
    return w  # Q*

# ---------- 5) Dự đoán ensemble ----------
def predict_posterior(pool, weights, X_test, Y_base_test):
    M = len(pool)
    yhats = []
    for m in range(M):
        yhat = pool[m]["tree"].predict(X_test[:, pool[m]["feat"]])  # (n_test, n_targets)
        yhats.append(yhat)
    yhats = np.stack(yhats, axis=0)          # (M, n_test, n_targets)
    w = weights.reshape(M, 1, 1)
    y_pred_diff = (w * yhats).sum(axis=0)    # (n_test, n_targets)
    return Y_base_test + y_pred_diff

## Học trọng số

In [31]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    pack = preprocess(symbol, lag=7, val=0.1)
    X_train, y_train = pack["train"]
    X_val,   y_val   = pack["val"]
    X_test,  y_test  = pack["test"]  # (diff) — chỉ dùng để tham khảo
    Y_base_test = pack["Y_base"]["test"]
    Y_true_test = pack["Y_true"]["test"]

    # 1) Pool
    pool = build_pool(X_train, y_train, X_val, M=50, mtry=None, seed=42)

    # 2) Loss trên val (RMSE multi-output)
    losses = np.array([root_mean_squared_error(y_val, p["yhat_val"]) for p in pool], dtype=float)

    # 3) Prior P (chọn 1)
    # P = prior_uniform(len(pool))
    # P = prior_diversity(pool, alpha=0.6)
    P = prior_combo(pool, alpha=0.6, gamma=0.4)

    # 4) Gibbs posterior Q* (nghiệm đóng)
    Q = pac_bayes_weights(losses, P, lam=10.0)   # tune lam theo val (hoặc grid nhỏ)

    # (tuỳ chọn) Sparsify: lấy top-k theo Q để tăng tốc suy luận
    k = 25
    S_idx = np.argsort(-Q)[:k]
    Qk = Q[S_idx] / Q[S_idx].sum()
    pool_k = [pool[i] for i in S_idx]

    # 5) Dự đoán test
    Y_pred = predict_posterior(pool_k, Qk, X_test, Y_base_test)

    # 6) Đánh giá global

    r2   = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100
    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9360, MAPE: 0.8683
Symbol: BCM, R2: 0.9469, MAPE: 1.4182
Symbol: BID, R2: 0.8989, MAPE: 1.1028
Symbol: BVH, R2: 0.9736, MAPE: 1.1686
Symbol: CTG, R2: 0.9628, MAPE: 1.1571
Symbol: FPT, R2: 0.9890, MAPE: 1.1968
Symbol: GAS, R2: 0.9498, MAPE: 0.8103
Symbol: GVR, R2: 0.9634, MAPE: 1.7855
Symbol: HDB, R2: 0.9728, MAPE: 1.1347
Symbol: HPG, R2: 0.9048, MAPE: 1.0226
Symbol: LPB, R2: 0.9943, MAPE: 1.3767
Symbol: MBB, R2: 0.9502, MAPE: 1.1158
Symbol: MSN, R2: 0.9466, MAPE: 1.2232
Symbol: MWG, R2: 0.9800, MAPE: 1.3061
Symbol: PLX, R2: 0.9778, MAPE: 1.1783
Symbol: SAB, R2: 0.9224, MAPE: 0.9414
Symbol: SHB, R2: 0.9559, MAPE: 0.9898
Symbol: SSB, R2: 0.9541, MAPE: 0.9348
Symbol: SSI, R2: 0.9285, MAPE: 1.1559
Symbol: STB, R2: 0.9745, MAPE: 1.1783
Symbol: TCB, R2: 0.9769, MAPE: 1.1545
Symbol: TPB, R2: 0.9456, MAPE: 1.1617
Symbol: VCB, R2: 0.8595, MAPE: 0.7778
Symbol: VHM, R2: 0.9578, MAPE: 1.2338
Symbol: VIB, R2: 0.9261, MAPE: 0.9616
Symbol: VIC, R2: 0.9629, MAPE: 1.1728
Symbol: VJC,

## Không học trọng số

In [32]:
tracks = {"r2": [], "mape": []}
for symbol in VN30:
    pack = preprocess(symbol, lag=7, val=0.1)
    X_train, y_train = pack["train"]
    X_val,   y_val   = pack["val"]
    X_test,  y_test  = pack["test"]  # (diff) — chỉ dùng để tham khảo
    Y_base_test = pack["Y_base"]["test"]
    Y_true_test = pack["Y_true"]["test"]

    # 1) Pool
    pool = build_pool(X_train, y_train, X_val, M=50, mtry=None, seed=42)

    # 2) Loss trên val (RMSE multi-output)
    losses = np.array([root_mean_squared_error(y_val, p["yhat_val"]) for p in pool], dtype=float)

    # 3) Prior P (chọn 1)
    # P = prior_uniform(len(pool))
    # P = prior_diversity(pool, alpha=0.6)
    P = prior_combo(pool, alpha=0.6, gamma=0.4)

    # 4) Gibbs posterior Q* (nghiệm đóng)
    Q = pac_bayes_weights(losses, P, lam=10.0)   # tune lam theo val (hoặc grid nhỏ)

    # (tuỳ chọn) Sparsify: lấy top-k theo Q để tăng tốc suy luận
    k = 25
    S_idx = np.argsort(-Q)[:k]
    Qk = Q[S_idx] / Q[S_idx].sum()
    pool_k = [pool[i] for i in S_idx]

    # 5) Dự đoán test
    # Gán Qk trung bình đều
    Qk = np.full(k, 1.0/k)
    Y_pred = predict_posterior(pool_k, Qk, X_test, Y_base_test)

    # 6) Đánh giá global

    r2   = r2_score(Y_true_test, Y_pred)
    mape = mean_absolute_percentage_error(Y_true_test, Y_pred) * 100
    print(f"Symbol: {symbol}, R2: {r2:.4f}, MAPE: {mape:.4f}")

    tracks["r2"].append(r2)
    tracks["mape"].append(mape)

print(f"Mean R2: {np.mean(tracks['r2']):.4f}, Mean MAPE: {np.mean(tracks['mape']):.4f}")
print(f"Std R2: {np.std(tracks['r2']):.4f}, Std MAPE: {np.std(tracks['mape']):.4f}")

Symbol: ACB, R2: 0.9360, MAPE: 0.8684
Symbol: BCM, R2: 0.9470, MAPE: 1.4181
Symbol: BID, R2: 0.8989, MAPE: 1.1025
Symbol: BVH, R2: 0.9736, MAPE: 1.1686
Symbol: CTG, R2: 0.9628, MAPE: 1.1570
Symbol: FPT, R2: 0.9890, MAPE: 1.1962
Symbol: GAS, R2: 0.9498, MAPE: 0.8103
Symbol: GVR, R2: 0.9634, MAPE: 1.7855
Symbol: HDB, R2: 0.9728, MAPE: 1.1347
Symbol: HPG, R2: 0.9048, MAPE: 1.0227
Symbol: LPB, R2: 0.9943, MAPE: 1.3769
Symbol: MBB, R2: 0.9502, MAPE: 1.1159
Symbol: MSN, R2: 0.9466, MAPE: 1.2237
Symbol: MWG, R2: 0.9800, MAPE: 1.3055
Symbol: PLX, R2: 0.9778, MAPE: 1.1784
Symbol: SAB, R2: 0.9224, MAPE: 0.9412
Symbol: SHB, R2: 0.9559, MAPE: 0.9897
Symbol: SSB, R2: 0.9541, MAPE: 0.9346
Symbol: SSI, R2: 0.9284, MAPE: 1.1560
Symbol: STB, R2: 0.9745, MAPE: 1.1783
Symbol: TCB, R2: 0.9769, MAPE: 1.1545
Symbol: TPB, R2: 0.9456, MAPE: 1.1617
Symbol: VCB, R2: 0.8596, MAPE: 0.7778
Symbol: VHM, R2: 0.9578, MAPE: 1.2339
Symbol: VIB, R2: 0.9261, MAPE: 0.9617
Symbol: VIC, R2: 0.9630, MAPE: 1.1726
Symbol: VJC,